# Prompt 17: Hands-On Capstone Project

**HashiCorp Certified Terraform Associate (004) — End-to-End Project**

This capstone exercises all 8 exam objectives using only the `random`, `local`, and `null` providers — **no cloud credentials required**. Every task maps to one or more exam objectives.

---

## Project Overview

**Goal**: Provision a simulated application environment with:
- Multiple resources created with `for_each`
- A reusable child module that writes config files to disk
- Input validation and custom health checks
- Lifecycle protection on a critical resource
- A `moved` block for refactoring
- A backend configuration for remote state
- Complete workflow walkthrough

## Directory Structure

```
capstone/
├── main.tf              # Root module — resources, module calls, moved block
├── variables.tf         # Input variable declarations with validation
├── outputs.tf           # Output values
├── locals.tf            # Local value expressions
├── versions.tf          # required_providers and backend
└── modules/
    └── config-file/
        ├── main.tf      # local_file resource
        ├── variables.tf # Module inputs
        └── outputs.tf   # Module outputs
```

---

## Exam Objective Coverage

| Task | Exam Objective |
|------|----------------|
| 1 — required_providers | Objective 2: Terraform Fundamentals |
| 2 — Input variables | Objective 4: Terraform Configuration |
| 3 — Locals | Objective 4: Terraform Configuration |
| 4 — for_each with random_pet | Objective 4: Terraform Configuration |
| 5 — Child module | Objective 5: Terraform Modules |
| 6 — Module call + output access | Objective 5: Terraform Modules |
| 7 — Outputs (sensitive + for expression) | Objective 4: Terraform Configuration |
| 8 — validation + check block | Objective 4: Terraform Configuration |
| 9 — lifecycle prevent_destroy | Objective 4: Terraform Configuration |
| 10 — S3 backend configuration | Objective 6: State Management |
| 11 — moved block | Objective 6: State Management |
| 12 — Complete workflow | Objective 3: Core Terraform Workflow |

---

## Task 1: Root Module — `versions.tf`

**Exam Objective 2**: Terraform Fundamentals — providers, plugin model, required_providers

Pin `hashicorp/random` and `hashicorp/local` with `~>` (pessimistic constraint operator) so Terraform accepts patch updates but not breaking minor/major changes.

In [ ]:
# FILE: capstone/versions.tf
# ─────────────────────────────────────────────────────────────────────────────
# Objective 2: required_providers declares the source and version constraints
# for all providers used in this module. The ~> operator means:
#   ~> 3.0  →  >= 3.0, < 4.0  (allows 3.x patches and minors, not 4.x)
#   ~> 2.4  →  >= 2.4, < 3.0  (allows 2.4.x patches only)
# ─────────────────────────────────────────────────────────────────────────────

terraform {
  required_version = ">= 1.5.0"

  required_providers {
    random = {
      source  = "hashicorp/random"
      version = "~> 3.0"
    }
    local = {
      source  = "hashicorp/local"
      version = "~> 2.4"
    }
  }
}

**Key Points**:
- `required_version` restricts which Terraform CLI versions can run this config
- `source` uses the `registry.terraform.io/<namespace>/<provider>` format (namespace/provider)
- `~> 3.0` locks to the 3.x series — `terraform init` will not select 4.x even if available
- After `terraform init`, a `.terraform.lock.hcl` file is created recording the exact version selected

**Expected `terraform init` output** (excerpt):
```
Initializing provider plugins...
- Finding hashicorp/random versions matching "~> 3.0"...
- Finding hashicorp/local versions matching "~> 2.4"...
- Installing hashicorp/random v3.6.1...
- Installing hashicorp/local v2.4.1...

Terraform has been successfully initialized!
```

---

## Task 2: Input Variables — `variables.tf`

**Exam Objective 4**: Terraform Configuration — variables, validation blocks, type constraints

In [ ]:
# FILE: capstone/variables.tf
# ─────────────────────────────────────────────────────────────────────────────
# Objective 4: Input variables — type constraints, default values, descriptions,
# and validation blocks with custom error messages.
# ─────────────────────────────────────────────────────────────────────────────

variable "environment" {
  type        = string
  description = "Deployment environment. Must be dev, staging, or prod."

  # Task 8: validation block — enforces allowed values with a human-readable error
  validation {
    condition     = contains(["dev", "staging", "prod"], var.environment)
    error_message = "The environment variable must be one of: dev, staging, prod."
  }
}

variable "app_name" {
  type        = string
  description = "Name of the application being deployed."
}

variable "instance_count" {
  type        = number
  description = "Number of application instances to simulate."
  default     = 2

  validation {
    condition     = var.instance_count >= 1 && var.instance_count <= 10
    error_message = "instance_count must be between 1 and 10."
  }
}

**Key Points**:
- The `validation` block runs when the variable value is set — before any resources are created
- `condition` must be a boolean expression using `var.<name>` to reference the variable being validated
- `error_message` must be a non-empty string and should clearly explain what is valid
- `default` makes a variable optional — without a default, the variable is **required**
- Type `number` accepts both integer and float values

**What happens when validation fails**:
```
╷
│ Error: Invalid value for variable
│
│   on variables.tf line 5:
│    5: variable "environment" {
│     ├────────────────
│     │ var.environment is "production"
│
│ The environment variable must be one of: dev, staging, prod.
╵
```

---

## Task 3: Locals Block — `locals.tf`

**Exam Objective 4**: Terraform Configuration — local values, expressions, string interpolation

In [ ]:
# FILE: capstone/locals.tf
# ─────────────────────────────────────────────────────────────────────────────
# Objective 4: Local values — computed expressions that can be referenced
# throughout the module as local.<name>. They are evaluated once and cached.
# Unlike variables, locals cannot be overridden by the caller.
# ─────────────────────────────────────────────────────────────────────────────

locals {
  # Build a consistent name prefix for all resources in this environment
  name_prefix = "${var.app_name}-${var.environment}"

  # Generate instance names by combining the prefix with index numbers
  instance_names = toset([
    for i in range(var.instance_count) : "${local.name_prefix}-${format("%02d", i + 1)}"
  ])

  # Common tags applied to all resources
  common_tags = {
    app         = var.app_name
    environment = var.environment
    managed_by  = "terraform"
  }
}

**Key Points**:
- Locals are referenced as `local.<name>` (singular `local`, not `locals`)
- Locals can reference variables (`var.*`), other locals (`local.*`), and resource attributes
- `range(n)` generates `[0, 1, ..., n-1]`; `format("%02d", n)` zero-pads to 2 digits
- `toset()` converts the list to a set (required for `for_each`)

**Example output** for `app_name = "myapp"`, `environment = "dev"`, `instance_count = 2`:
```
local.name_prefix     = "myapp-dev"
local.instance_names  = toset(["myapp-dev-01", "myapp-dev-02"])
```

---

## Task 4: Resources with `for_each` — `main.tf` (Part 1)

**Exam Objective 4**: Terraform Configuration — `for_each`, resource addressing, `toset()`

In [ ]:
# FILE: capstone/main.tf  (Part 1 — random_pet resources)
# ─────────────────────────────────────────────────────────────────────────────
# Objective 4: for_each creates one resource instance per element in the set.
# Each instance is keyed by the set element value (the instance name string).
# Access: random_pet.instances["myapp-dev-01"].id
# ─────────────────────────────────────────────────────────────────────────────

resource "random_pet" "instances" {
  for_each = local.instance_names

  # Use the instance name as a keepsake to differentiate instances
  keepers = {
    instance = each.key
  }

  length    = 2
  separator = "-"
}

# Task 9: lifecycle prevent_destroy — Terraform will refuse to destroy
# the primary instance, protecting a critical resource from accidental deletion.
resource "random_id" "primary_token" {
  byte_length = 16
  keepers = {
    environment = var.environment
    app         = var.app_name
  }

  lifecycle {
    prevent_destroy = true
  }
}

**Key Points**:
- `for_each` on a **set of strings** uses each string as both the key (`each.key`) and value (`each.value`)
- `for_each` on a **map** uses map keys as `each.key` and map values as `each.value`
- Resources are addressed as `resource_type.resource_name["key"]` — e.g., `random_pet.instances["myapp-dev-01"]`
- `keepers` in the `random_*` providers forces regeneration when the keepers map changes
- `prevent_destroy = true` causes Terraform to **error and abort** any plan that includes destroying this resource

**State list output** (example):
```
random_id.primary_token
random_pet.instances["myapp-dev-01"]
random_pet.instances["myapp-dev-02"]
```

---

## Task 5: Child Module — `modules/config-file/`

**Exam Objective 5**: Terraform Modules — creating modules, variable scope, outputs

In [ ]:
# FILE: capstone/modules/config-file/variables.tf
# ─────────────────────────────────────────────────────────────────────────────
# Objective 5: Module inputs are declared as variable blocks inside the module.
# The caller passes values via arguments in the module block.
# These variables are PRIVATE to this module — the root module cannot access
# them; only the module's own resources can use them.
# ─────────────────────────────────────────────────────────────────────────────

variable "filename" {
  type        = string
  description = "Absolute or relative path where the config file will be written."
}

variable "content" {
  type        = string
  description = "Text content to write into the config file."
}

variable "app_name" {
  type        = string
  description = "Application name — used in the config file header."
}

In [ ]:
# FILE: capstone/modules/config-file/main.tf
# ─────────────────────────────────────────────────────────────────────────────
# The local_file resource writes content to disk on the machine running Terraform.
# This simulates writing application configuration files.
# ─────────────────────────────────────────────────────────────────────────────

resource "local_file" "config" {
  filename = var.filename
  content  = <<-EOT
    # Configuration for ${var.app_name}
    # Generated by Terraform — do not edit manually
    ${var.content}
  EOT

  # Ensure the file gets a predictable permission set
  file_permission = "0644"
}

In [ ]:
# FILE: capstone/modules/config-file/outputs.tf
# ─────────────────────────────────────────────────────────────────────────────
# Objective 5: Module outputs expose selected values to the calling module.
# Only what is declared in outputs.tf is accessible from outside this module.
# The caller accesses these as: module.<module_name>.<output_name>
# ─────────────────────────────────────────────────────────────────────────────

output "file_path" {
  description = "The absolute path of the written config file."
  value       = local_file.config.filename
}

output "content_checksum" {
  description = "MD5 checksum of the written file content."
  value       = local_file.config.content_md5
}

**Key Points**:
- Every Terraform configuration is a module — the `modules/config-file/` directory is a **child module**
- Module inputs (`variable` blocks) are scoped to the module — root module variables are NOT automatically inherited
- Module outputs (`output` blocks) are the only values exposed to the calling module
- Resources inside a module are NOT directly addressable from outside: `local_file.config` is private
- Standard module structure: `main.tf`, `variables.tf`, `outputs.tf`, `README.md`, `versions.tf`

---

## Task 6: Calling the Child Module — `main.tf` (Part 2)

**Exam Objective 5**: Terraform Modules — module blocks, passing arguments, accessing outputs

In [ ]:
# FILE: capstone/main.tf  (Part 2 — module call)
# ─────────────────────────────────────────────────────────────────────────────
# Objective 5: The module block calls the child module.
# - source: local path (requires ./ prefix)
# - All other arguments are input variable values for the module
# ─────────────────────────────────────────────────────────────────────────────

module "app_config" {
  source = "./modules/config-file"

  # These arguments map directly to the module's input variable names:
  app_name = var.app_name
  filename = "${path.module}/generated/${local.name_prefix}-config.ini"
  content  = <<-EOT
    app_name       = ${var.app_name}
    environment    = ${var.environment}
    instance_count = ${var.instance_count}
    token          = ${random_id.primary_token.hex}
  EOT
}

# Task 8: check block (Terraform 1.5+)
# Runs a continuous validation assertion to verify config file was created
check "config_file_exists" {
  assert {
    condition     = fileexists(module.app_config.file_path)
    error_message = "Config file was not created at ${module.app_config.file_path}."
  }
}

**Key Points**:
- **Local module source**: must start with `./` or `../` — a bare path like `modules/config-file` is not valid
- **Accessing module outputs**: `module.app_config.file_path` (format: `module.<name>.<output>`)
- `path.module` is a built-in expression that resolves to the directory containing the current `.tf` file
- **`check` block**: runs after apply as a health assertion; a failing check shows a warning but does NOT fail the apply
- `check` blocks are evaluated on every `terraform plan` and `terraform apply` — and by HCP Terraform's health assessments

---

## Task 7: Output Values — `outputs.tf`

**Exam Objective 4**: Terraform Configuration — outputs, sensitive outputs, `for` expressions

In [ ]:
# FILE: capstone/outputs.tf
# ─────────────────────────────────────────────────────────────────────────────
# Objective 4: Output values expose information after apply.
# sensitive = true prevents the value from appearing in plan/apply output
# and in the CLI output unless explicitly requested.
# ─────────────────────────────────────────────────────────────────────────────

# Sensitive output — token should not appear in terminal output
output "primary_token" {
  description = "The primary application token (sensitive — do not log)."
  value       = random_id.primary_token.hex
  sensitive   = true
}

# Non-sensitive output using a for expression
# Collects the generated pet name (random_pet.instances[*].id) into a map
output "instance_names" {
  description = "Map of instance names to their generated random pet names."
  value = {
    for k, v in random_pet.instances : k => v.id
  }
}

# Module output re-exported from root module
output "config_file_path" {
  description = "Path to the generated application config file."
  value       = module.app_config.file_path
}

output "config_checksum" {
  description = "MD5 checksum of the config file for drift detection."
  value       = module.app_config.content_checksum
}

**Key Points**:
- `sensitive = true` masks the value in `terraform plan`, `terraform apply`, and `terraform output`
- View a sensitive output explicitly: `terraform output -json primary_token` (or `terraform output primary_token`)
- **`for` expression syntax**: `{ for k, v in <collection> : <key_expr> => <value_expr> }`
  - Produces a map where each key is `k` and each value is `v.id`
- Root module outputs appear in `terraform output` after apply

**Expected `terraform output` (excerpt)**:
```
config_checksum = "a1b2c3d4e5f6..."
config_file_path = "./generated/myapp-dev-config.ini"
instance_names = {
  "myapp-dev-01" = "adapted-grizzly"
  "myapp-dev-02" = "vocal-coyote"
}
primary_token = <sensitive>
```

---

## Task 8: Validation and `check` Block

**Exam Objective 4**: Terraform Configuration — `validation`, `check` blocks, continuous validation

*(The `validation` block is already in `variables.tf` from Task 2. The `check` block is in `main.tf` from Task 6.)*

Here is a consolidated view of how validation and checks work together:

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VALIDATION vs CHECK BLOCK — KEY EXAM DIFFERENCES
# ─────────────────────────────────────────────────────────────────────────────

# 1. Variable validation block — runs BEFORE any resources are created
#    Fails the entire apply if the condition is false
#    Can ONLY reference var.<name> (the variable being validated)
variable "environment" {
  type = string
  validation {
    condition     = contains(["dev", "staging", "prod"], var.environment)
    error_message = "environment must be dev, staging, or prod."
  }
}

# 2. check block — runs AFTER resources are applied
#    Reports a WARNING if the assertion fails (does not stop the apply)
#    Can reference any resource, data source, module output, or expression
#    Evaluated on every plan/apply and in HCP Terraform health assessments
check "config_file_exists" {
  assert {
    condition     = fileexists(module.app_config.file_path)
    error_message = "Config file was not found at expected path."
  }
}

# 3. precondition / postcondition inside a resource lifecycle block
#    precondition: evaluated BEFORE resource is created/updated — hard failure
#    postcondition: evaluated AFTER resource is created/updated — hard failure
resource "random_id" "example" {
  byte_length = 8

  lifecycle {
    precondition {
      condition     = var.environment != ""
      error_message = "environment cannot be empty when creating a token."
    }

    postcondition {
      condition     = length(self.hex) == 16
      error_message = "Generated token is not the expected length of 16 hex chars."
    }
  }
}

**Comparison Table**:

| Feature | `validation` block | `check` block | `precondition` / `postcondition` |
|---|---|---|---|
| Location | Inside `variable` block | Top-level in module | Inside `resource` `lifecycle` |
| When evaluated | Before plan/apply | After apply | During resource create/update |
| Failure behavior | **Hard error** — stops everything | **Warning** — apply continues | **Hard error** — stops apply |
| Can reference resources? | No — only `var.<name>` | Yes — any expression | Yes — `self.*` in postcondition |
| Available since | Terraform 0.13 | Terraform 1.5 | Terraform 1.2 |

---

## Task 9: `lifecycle` Block with `prevent_destroy`

**Exam Objective 4**: Terraform Configuration — lifecycle meta-arguments

*(Already defined in Task 4's `main.tf`. Shown here with full explanation.)*

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LIFECYCLE ARGUMENTS — COMPLETE REFERENCE
# ─────────────────────────────────────────────────────────────────────────────

resource "random_id" "primary_token" {
  byte_length = 16
  keepers = {
    environment = var.environment
    app         = var.app_name
  }

  lifecycle {
    # Prevents terraform destroy and any plan that would delete this resource.
    # The only way to destroy it is to remove prevent_destroy, then re-apply.
    prevent_destroy = true

    # create_before_destroy — new resource is created before the old one is
    # destroyed. Critical for resources that must not have availability gaps.
    # create_before_destroy = true

    # ignore_changes — Terraform won't update the resource when these
    # attributes change in the configuration. Useful for fields managed
    # by the cloud provider directly (e.g., auto-assigned tags).
    # ignore_changes = [keepers]

    # replace_triggered_by — force replacement of THIS resource when the
    # referenced resource or attribute changes.
    # replace_triggered_by = [random_pet.instances]
  }
}

# ─────────────────────────────────────────────────────────────────────────────
# WHAT HAPPENS WHEN YOU TRY TO DESTROY A prevent_destroy RESOURCE:
# ─────────────────────────────────────────────────────────────────────────────
#
# $ terraform destroy
#
# ╷
# │ Error: Instance cannot be destroyed
# │
# │   on main.tf line 22:
# │   22: resource "random_id" "primary_token" {
# │
# │ Resource random_id.primary_token has lifecycle.prevent_destroy set, but
# │ the plan calls for this resource to be destroyed. To avoid this error
# │ and continue, remove the prevent_destroy lifecycle configuration or
# │ remove the resource from the Terraform configuration.
# ╵
print("See comments above for the prevent_destroy error message")

---

## Task 10: S3 Backend Configuration

**Exam Objective 6**: State Management — backends, state locking, remote state

This HCL is written for documentation purposes — it requires valid AWS credentials and an existing S3 bucket and DynamoDB table to execute.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# S3 BACKEND CONFIGURATION — For reference/study (requires AWS credentials)
# This would replace the terraform {} block in versions.tf to enable
# remote state storage in S3 with DynamoDB locking.
# ─────────────────────────────────────────────────────────────────────────────

# Add this to capstone/versions.tf to enable remote state:
'''
terraform {
  required_version = ">= 1.5.0"

  required_providers {
    random = {
      source  = "hashicorp/random"
      version = "~> 3.0"
    }
    local = {
      source  = "hashicorp/local"
      version = "~> 2.4"
    }
  }

  backend "s3" {
    # S3 bucket for storing the state file
    bucket = "my-company-terraform-state"     # Must already exist
    key    = "capstone/terraform.tfstate"      # Path within the bucket
    region = "us-east-1"

    # Encrypt state at rest using S3 server-side encryption (SSE)
    encrypt = true

    # DynamoDB table for state locking
    # Table must have a partition key named "LockID" (type: String)
    dynamodb_table = "terraform-state-locks"

    # Optional: use an IAM role for cross-account access
    # role_arn = "arn:aws:iam::123456789012:role/TerraformRole"
  }
}
'''

print("S3 backend HCL reference shown above as a string (no AWS credentials needed to study)")

**Key Exam Points**:

| S3 Backend Component | Purpose |
|---|---|
| `bucket` | S3 bucket name — must already exist before `terraform init` |
| `key` | Path within the bucket where `terraform.tfstate` is stored |
| `region` | AWS region of the S3 bucket |
| `encrypt = true` | Enables SSE (server-side encryption) for the state file |
| `dynamodb_table` | DynamoDB table name for state locking — prevents concurrent writes |

**Migration workflow** (local → S3 backend):
```bash
# 1. Add backend block to versions.tf
# 2. Re-run init — Terraform detects backend change and prompts to migrate:
terraform init

# Terraform output:
# Do you want to copy existing state to the new backend?
# yes
# Successfully configured the backend "s3"!
```

**Critical**: Backend configuration **cannot use variable references** (e.g., `bucket = var.state_bucket` is invalid). Backend config is evaluated before variables during `terraform init`.

---

## Task 11: `moved` Block — `main.tf` (Part 3)

**Exam Objective 6**: State Management — `moved` block, refactoring without destroy/recreate

In [ ]:
# FILE: capstone/main.tf  (Part 3 — moved block)
# ─────────────────────────────────────────────────────────────────────────────
# Objective 6: The moved block tells Terraform that a resource was renamed
# in configuration but should NOT be destroyed and recreated — instead,
# Terraform updates the state file to use the new address.
#
# Use case: You renamed random_pet.app_server to random_pet.primary_server
# Without moved: Terraform would destroy the old and create a new resource
# With moved: Terraform just updates the state entry — no infrastructure change
# ─────────────────────────────────────────────────────────────────────────────

# Scenario: We previously had this resource:
# resource "random_id" "old_name" { ... }
# And renamed it to random_id.new_name in the configuration.
# This moved block prevents destroy/recreate:

moved {
  from = random_id.old_name
  to   = random_id.new_name
}

# The new resource (after rename):
resource "random_id" "new_name" {
  byte_length = 8
  keepers = {
    app = var.app_name
  }
}

# ─────────────────────────────────────────────────────────────────────────────
# MOVED BLOCK — PLAN OUTPUT:
# ─────────────────────────────────────────────────────────────────────────────
# Terraform will perform the following actions:
#
#   # random_id.old_name has moved to random_id.new_name
#     resource "random_id" "new_name" {
#         id          = "abcdef12"
#         # (1 unchanged attribute hidden)
#     }
#
# Plan: 0 to add, 0 to change, 0 to destroy.
# ─────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────
# REMOVED BLOCK — stop managing a resource WITHOUT destroying it:
# ─────────────────────────────────────────────────────────────────────────────
# removed {
#   from = random_pet.decommissioned_instance
#   lifecycle {
#     destroy = false  # false = remove from state, do NOT destroy the resource
#   }
# }

print("moved and removed block examples shown in comments above")

**Key Exam Points**:
- `moved` block is the **preferred** way to rename resources (vs `terraform state mv` which is imperative)
- `moved` blocks are stored in `.tf` files — they're version-controlled and reviewable
- After the state is updated, you can **remove** the `moved` block (but it's safe to keep it)
- `removed` block with `destroy = false`: removes from state without destroying real infrastructure
- `terraform state mv` (older approach): does the same thing imperatively from CLI, not stored in config

---

## Task 12: Complete Terraform Workflow

**Exam Objective 3**: Core Terraform Workflow — all CLI commands from init to destroy

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# COMPLETE TERRAFORM WORKFLOW — capstone project
# Run these commands from the capstone/ directory
# ─────────────────────────────────────────────────────────────────────────────

workflow_steps = [
    {
        "step": "1. terraform init",
        "command": "terraform init",
        "purpose": "Download providers (random ~>3.0, local ~>2.4), initialize backend, create .terraform/",
        "key_output": "Terraform has been successfully initialized!",
        "objective": "2 — providers, plugin model"
    },
    {
        "step": "2. terraform fmt",
        "command": "terraform fmt -recursive",
        "purpose": "Format all .tf files to canonical style. -recursive includes modules/",
        "key_output": "Lists files that were reformatted (empty output = already formatted)",
        "objective": "3 — core workflow, formatting"
    },
    {
        "step": "3. terraform validate",
        "command": "terraform validate",
        "purpose": "Check syntax, required args, type constraints, reference validity. Does NOT contact cloud APIs.",
        "key_output": "Success! The configuration is valid.",
        "objective": "3 — core workflow"
    },
    {
        "step": "4. terraform plan (saved)",
        "command": "terraform plan -var='app_name=myapp' -var='environment=dev' -out=plan.tfplan",
        "purpose": "Preview changes; save plan to file so apply executes exactly this plan.",
        "key_output": "Plan: 5 to add, 0 to change, 0 to destroy. Saved to: plan.tfplan",
        "objective": "3 — plan output, saved plans"
    },
    {
        "step": "5. terraform apply (from saved plan)",
        "command": "terraform apply plan.tfplan",
        "purpose": "Apply the exact plan that was saved — no confirmation prompt needed.",
        "key_output": "Apply complete! Resources: 5 added, 0 changed, 0 destroyed.",
        "objective": "3 — apply, idempotency"
    },
    {
        "step": "6. terraform state list",
        "command": "terraform state list",
        "purpose": "List all resources currently tracked in state.",
        "key_output": "module.app_config.local_file.config\nrandom_id.new_name\nrandom_id.primary_token\nrandom_pet.instances[\"myapp-dev-01\"]\nrandom_pet.instances[\"myapp-dev-02\"]",
        "objective": "7 — inspecting state"
    },
    {
        "step": "7. terraform state show",
        "command": 'terraform state show \'random_id.primary_token\'',
        "purpose": "Show all attributes of a specific resource from state.",
        "key_output": "# random_id.primary_token:\nresource \"random_id\" \"primary_token\" {\n  b64_std = \"...\"\n  hex     = \"a1b2c3d4e5f6...\"\n  id      = \"...\"\n}",
        "objective": "7 — inspecting state"
    },
    {
        "step": "8. terraform output",
        "command": "terraform output",
        "purpose": "Show all output values. Sensitive outputs show as <sensitive>.",
        "key_output": "config_checksum = \"abc123...\"\nconfig_file_path = \"./generated/myapp-dev-config.ini\"\ninstance_names = { ... }\nprimary_token = <sensitive>",
        "objective": "7 — outputs"
    },
    {
        "step": "9. terraform output (specific, JSON)",
        "command": "terraform output -json instance_names",
        "purpose": "Show a specific output in JSON format — useful for scripting.",
        "key_output": '{"myapp-dev-01": "adapted-grizzly", "myapp-dev-02": "vocal-coyote"}',
        "objective": "7 — outputs"
    },
    {
        "step": "10. terraform console",
        "command": "terraform console",
        "purpose": "Interactive expression evaluator — test functions, inspect local values.",
        "key_output": '> local.name_prefix\n\"myapp-dev\"\n> join("-", ["a","b"])\n\"a-b\"',
        "objective": "7 — terraform console"
    },
    {
        "step": "11. terraform plan -refresh-only",
        "command": "terraform plan -refresh-only",
        "purpose": "Check for drift — updates state to match real infra. Does NOT apply config changes.",
        "key_output": "No changes. Your infrastructure matches the configuration.",
        "objective": "6 — drift detection"
    },
    {
        "step": "12. terraform destroy",
        "command": "# Remove prevent_destroy first, then:",
        "extra_command": "terraform destroy -var='app_name=myapp' -var='environment=dev'",
        "purpose": "Destroy all managed resources. Will fail if prevent_destroy = true is still set.",
        "key_output": "Destroy complete! Resources: 5 destroyed.",
        "objective": "3 — destroy = apply -destroy"
    },
]

for step in workflow_steps:
    print(f"{'='*60}")
    print(f"Step: {step['step']}")
    print(f"Command: {step['command']}")
    if 'extra_command' in step:
        print(f"         {step['extra_command']}")
    print(f"Purpose: {step['purpose']}")
    print(f"Objective: {step['objective']}")
    print()

---

## Complete `main.tf` — Final Assembly

All parts of `main.tf` assembled into a single file reference:

In [ ]:
# FILE: capstone/main.tf  (COMPLETE FILE)
# ─────────────────────────────────────────────────────────────────────────────

# ── Task 4: random_pet resources with for_each ─────────────────────────────
resource "random_pet" "instances" {
  for_each = local.instance_names

  keepers = {
    instance = each.key
  }

  length    = 2
  separator = "-"
}

# ── Task 9: lifecycle prevent_destroy ──────────────────────────────────────
resource "random_id" "primary_token" {
  byte_length = 16
  keepers = {
    environment = var.environment
    app         = var.app_name
  }

  lifecycle {
    prevent_destroy = true
  }
}

# ── Task 11: moved block ────────────────────────────────────────────────────
moved {
  from = random_id.old_name
  to   = random_id.new_name
}

resource "random_id" "new_name" {
  byte_length = 8
  keepers = {
    app = var.app_name
  }
}

# ── Task 6: call the child module ──────────────────────────────────────────
module "app_config" {
  source = "./modules/config-file"

  app_name = var.app_name
  filename = "${path.module}/generated/${local.name_prefix}-config.ini"
  content  = <<-EOT
    app_name       = ${var.app_name}
    environment    = ${var.environment}
    instance_count = ${var.instance_count}
    token          = ${random_id.primary_token.hex}
  EOT
}

# ── Task 8: check block ────────────────────────────────────────────────────
check "config_file_exists" {
  assert {
    condition     = fileexists(module.app_config.file_path)
    error_message = "Config file was not created at ${module.app_config.file_path}."
  }
}

---

## Capstone Real-World Extension Scenarios

<details><summary>Scenario 1: Handling a `prevent_destroy` Error During an Emergency Scale-Down</summary>

**Problem**: The operations team decides to decommission the `random_id.primary_token` resource as part of a system migration. When they run `terraform destroy`, Terraform errors:

```
Error: Instance cannot be destroyed

  on main.tf line 22:
  22: resource "random_id" "primary_token" {

Resource random_id.primary_token has lifecycle.prevent_destroy set,
but the plan calls for this resource to be destroyed.
```

**Resolution Steps**:

```hcl
# Step 1: Edit main.tf — change prevent_destroy to false (or remove the lifecycle block)
resource "random_id" "primary_token" {
  byte_length = 16
  keepers = {
    environment = var.environment
    app         = var.app_name
  }

  lifecycle {
    prevent_destroy = false  # Changed from true
  }
}
```

```bash
# Step 2: Re-run terraform plan to verify the resource is now destroyable
terraform plan -var='app_name=myapp' -var='environment=dev'

# Step 3: Apply the change (which removes the lifecycle protection)
# Note: this plan has 0 infrastructure changes — only the lifecycle meta-argument changed
terraform apply -var='app_name=myapp' -var='environment=dev'

# Step 4: Now destroy
terraform destroy -var='app_name=myapp' -var='environment=dev'
```

**Expected Outcome**: After removing `prevent_destroy`, the destroy plan proceeds and the resource is deleted.

**Exam Sub-Objective**: Lifecycle meta-arguments — `prevent_destroy` behavior and how to safely override it (Objective 4).

</details>

---

<details><summary>Scenario 2: Migrating the Local Backend to S3</summary>

**Problem**: The capstone project was initially using the default local backend (state stored in `terraform.tfstate`). The team is growing and needs to share state. They want to migrate to S3 with DynamoDB locking.

**Steps**:

```bash
# Before migration — state is in local terraform.tfstate:
terraform state list
# random_id.primary_token
# random_pet.instances["myapp-dev-01"]
# ...
```

```hcl
# Step 1: Add the backend block to versions.tf:
terraform {
  backend "s3" {
    bucket         = "my-company-terraform-state"
    key            = "capstone/terraform.tfstate"
    region         = "us-east-1"
    encrypt        = true
    dynamodb_table = "terraform-state-locks"
  }
}
```

```bash
# Step 2: Run terraform init — Terraform detects the backend change:
terraform init

# Output:
# Initializing the backend...
# Do you want to copy existing state to the new backend?
#   Pre-existing state was found while migrating the previous "local" backend to the
#   newly configured "s3" backend. No existing state was found in the newly configured
#   "s3" backend. Do you want to copy this state to the new "s3" backend?
# yes
#
# Successfully configured the backend "s3"!

# Step 3: Verify state is now in S3
terraform state list  # Should return same resources, but state now lives in S3

# Step 4: Remove the local terraform.tfstate file (it's now in S3)
# rm terraform.tfstate terraform.tfstate.backup
```

**Expected Outcome**: State is migrated from `terraform.tfstate` on disk to the S3 bucket. Future runs read/write state from S3 with DynamoDB locking.

**Exam Sub-Objective**: Backend migration — `terraform init` after backend change, state migration prompts (Objective 6).

</details>

---

<details><summary>Scenario 3: Renaming a `for_each` Resource Key</summary>

**Problem**: The team wants to rename instance keys from `"myapp-dev-01"` to `"instance-01"` to make them environment-agnostic. Without a `moved` block, Terraform would destroy the instances with the old keys and create new ones with the new keys.

**Configuration Change**:

```hcl
# Old locals.tf:
# instance_names = toset([
#   for i in range(var.instance_count) : "${local.name_prefix}-${format("%02d", i + 1)}"
# ])
# → Creates: random_pet.instances["myapp-dev-01"], random_pet.instances["myapp-dev-02"]

# New locals.tf (environment-agnostic keys):
locals {
  instance_names = toset([
    for i in range(var.instance_count) : "instance-${format("%02d", i + 1)}"
  ])
}
# → Creates: random_pet.instances["instance-01"], random_pet.instances["instance-02"]
```

```hcl
# Add moved blocks to main.tf for each renamed instance:
moved {
  from = random_pet.instances["myapp-dev-01"]
  to   = random_pet.instances["instance-01"]
}

moved {
  from = random_pet.instances["myapp-dev-02"]
  to   = random_pet.instances["instance-02"]
}
```

```bash
terraform plan -var='app_name=myapp' -var='environment=dev'

# Plan output:
#   # random_pet.instances["myapp-dev-01"] has moved to random_pet.instances["instance-01"]
#   # random_pet.instances["myapp-dev-02"] has moved to random_pet.instances["instance-02"]
# Plan: 0 to add, 0 to change, 0 to destroy.
```

**Expected Outcome**: Terraform updates state entries with no infrastructure changes. The random pets keep their generated names.

**Exam Sub-Objective**: `moved` block with `for_each` resource instances — refactoring without destroy/recreate (Objective 6).

</details>

---

<details><summary>Scenario 4: Debugging Why the Config File Wasn't Created</summary>

**Problem**: After a `terraform apply`, the `check "config_file_exists"` reports a warning that the config file is not at the expected path. The engineer needs to debug what happened.

**Debugging Steps**:

```bash
# Step 1: Check what path Terraform thinks the file should be at
terraform output config_file_path
# Output: "./generated/myapp-dev-config.ini"

# Step 2: Verify the file doesn't exist
ls ./generated/
# ls: cannot access './generated/': No such file or directory

# Step 3: Enable verbose logging to see what the local_file resource did
# (on Windows: set TF_LOG=DEBUG)
TF_LOG=DEBUG terraform apply -var='app_name=myapp' -var='environment=dev' 2>&1 | grep -i 'local_file'

# Step 4: Check state to see what Terraform recorded
terraform state show module.app_config.local_file.config
# Shows: filename = "./generated/myapp-dev-config.ini"

# Root cause: The 'generated/' directory doesn't exist and local_file
# does not create parent directories on some Terraform/provider versions.
```

```hcl
# Fix: Create the directory first using a null_resource or local-exec provisioner,
# OR use an absolute path where the directory already exists.

# Updated module call with existing directory:
module "app_config" {
  source   = "./modules/config-file"
  app_name = var.app_name
  filename = "${path.module}/${var.app_name}-${var.environment}-config.ini"  # No subdir
  content  = "..."
}
```

**TF_LOG level guidance**:
- `ERROR`: Only critical errors
- `WARN`: Warnings and errors
- `INFO`: General operational information
- `DEBUG`: Provider API calls, resource decisions
- `TRACE`: Everything including raw HTTP request/response bodies

**Exam Sub-Objective**: Verbose logging (`TF_LOG`, `TF_LOG_PATH`), `check` block warnings, `terraform state show` (Objective 7).

</details>

---

<details><summary>Scenario 5: Importing an Existing Resource into the Capstone</summary>

**Problem**: A `random_id` resource was manually created outside Terraform using a seed script. The team wants to bring it under Terraform management without recreating it.

**Using the `import` block (preferred, Terraform 1.5+)**:

```hcl
# Step 1: Add an import block to main.tf
# The 'id' for random_id is the hex-encoded value of the random bytes
import {
  to = random_id.legacy_token
  id = "deadbeef12345678"  # The hex value of the existing random_id
}
```

```bash
# Step 2: Run plan with config generation to create the resource block automatically
terraform plan \
  -var='app_name=myapp' \
  -var='environment=dev' \
  -generate-config-out=imported.tf

# Terraform generates imported.tf with:
# resource "random_id" "legacy_token" {
#   byte_length = 8
#   keepers     = null
# }

# Step 3: Review the generated configuration, adjust as needed, then apply
terraform apply -var='app_name=myapp' -var='environment=dev'
# random_id.legacy_token: Importing... [id=deadbeef12345678]
# random_id.legacy_token: Import complete [id=deadbeef12345678]
# Apply complete! Resources: 0 added, 0 changed, 0 destroyed.
# (1 imported)

# Step 4: Verify it's in state
terraform state show random_id.legacy_token
```

**Why NOT use `terraform import` CLI here**:
- `terraform import random_id.legacy_token deadbeef12345678` would write to state but NOT generate `imported.tf`
- You'd have to write the resource block manually before running the CLI import
- The `import` block approach shows the import in the plan for team review before applying

**Expected Outcome**: The existing `random_id` is now tracked in Terraform state with no infrastructure changes.

**Exam Sub-Objective**: `import` block vs `terraform import` CLI — config generation, plan visibility, preferred approach (Objective 7).

</details>

---

## Capstone Summary — All 8 Objectives Covered

| Objective | Coverage in Capstone |
|---|---|
| **1 — IaC with Terraform** | Project goal: version-controlled, reproducible infrastructure; declarative config |
| **2 — Terraform Fundamentals** | `required_providers`, `~>` version constraints, `.terraform.lock.hcl`, provider plugins |
| **3 — Core Workflow** | Full workflow: `init` → `fmt` → `validate` → `plan -out` → `apply` → `state list` → `output` → `destroy` |
| **4 — Terraform Configuration** | Variables, validation, locals, `for_each`, `for` expression, sensitive outputs, `lifecycle`, `check` block, `precondition`/`postcondition` |
| **5 — Terraform Modules** | Child module with standard structure, local source path, variable scope, output access |
| **6 — State Management** | S3 backend with DynamoDB locking, `moved` block, `removed` block, `-refresh-only`, backend migration |
| **7 — Maintaining Infrastructure** | `import` block with config generation, `state list`, `state show`, `terraform console`, `TF_LOG` debugging |
| **8 — HCP Terraform** | `cloud` block equivalent (S3 backend shown), remote state concepts, drift detection via health assessments |

---

## Quick CLI Reference Card

```
terraform init                     Initialize — download providers, set up backend
terraform init -upgrade            Upgrade providers to latest matching constraints
terraform fmt -recursive           Format all .tf files
terraform validate                 Check syntax/consistency (no cloud API calls)
terraform plan                     Preview changes
terraform plan -out=plan.tfplan    Save plan to file
terraform plan -refresh-only       Check for drift (update state only)
terraform plan -destroy            Preview destruction of all resources
terraform apply                    Apply changes (prompts for confirmation)
terraform apply plan.tfplan        Apply saved plan (no confirmation prompt)
terraform apply -refresh-only      Apply drift update (state only, no infra changes)
terraform destroy                  Destroy all managed resources
terraform state list               List all resources in state
terraform state show <addr>        Show attributes of a specific resource
terraform state mv <from> <to>     Move resource in state (imperative)
terraform state rm <addr>          Remove resource from state (no destroy)
terraform state pull               Download raw state to stdout
terraform output                   Show all outputs
terraform output -json <name>      Show specific output in JSON
terraform console                  Interactive expression evaluator
terraform import <addr> <id>       Import existing resource to state (legacy CLI)
terraform force-unlock <lock-id>   Release stuck state lock (use with caution)
terraform graph                    Output DOT-format dependency graph
terraform login                    Authenticate to HCP Terraform
TF_LOG=TRACE terraform apply       Maximum verbosity logging
TF_LOG_PATH=./tf.log               Write logs to file
```